In [12]:
import sys
sys.path.insert(0, "..")
from app.orchestrator.context import build_static_context, get_static_context

from app.orchestrator.tools import (
    CombineResultsToolCallArgs,
    GenerateChartToolCallArgs,
    SubmitAnswerArgs,
    ToolError,
    chart_tool_call_standin,
    combine_results_call_standin,
    dry_run,
    generate_beeswarm,
    get_conversation_findings,
    get_measure_dax,
    get_page_info,
    run_bigquery_sql,
    search_docs,
    submit_answer,
)

In [8]:
static_context = build_static_context('anthropic', get_static_context())

In [10]:
print(static_context[0]['text'])

SYSTEM INSTRUCTIONS

You are a senior data professional supporting business stakeholders evaluating a machine learning reorder-recommendation model and its projected financial impact. Translate technical analysis -- SQL, DAX, model evaluation metrics, financial modeling -- into clear, direct language a business audience can act on.

TOOLS
BigQuery answers upstream/warehouse questions -- raw order and product features, pre-model data -- plus one post-model exception: prediction_explanations, row-level model predictions with their SHAP explanations (why individual predictions, or a product/aisle/department slice, scored as they did). DAX answers all other post-model, dashboard-displayed questions -- model evaluation metrics, financial impact. Excluding that exception, these domains don't overlap.

run_dax_query results are capped at 1000 rows -- prefer TOPN or a tighter filter over a query likely to return more.

GROUNDING
There is no live source for future data, only current and histori

{
  "name": "generate_chart",
  "input_schema": {
    "properties": {
      "source_ref": {
        "description": "The reference id printed alongside the run_bigquery_sql or run_dax_query result you want to chart -- e.g. 'ref_1'. Copy it exactly as shown in that tool's result; never invent one.",
        "type": "string"
      },
      "spec": {
        "anyOf": [
          {
            "description": "One bar per category. Fields: x_field (cat), y_field (num).\n\nExpects one row per category, already aggregated:\n    [{\"department\": \"produce\", \"orders\": 5231},\n     {\"department\": \"dairy\",   \"orders\": 4102}]\n\nKeep categories readable: departments (~21) or aisles (~134), not products\n(~50k). An un-aggregated query hits the 1,000-row cap and fails.\n\nBy default, bars plot in the order rows arrive from the query. Set\nsort_order to sort by y_field instead.",
            "properties": {
              "value_format": {
                "default": "auto",
                "d